# Unilingual Typographic Attack — Classification Performance

Each image has **two typographic text boxes**, both containing the same English attack word (same target class). Only English text is present.

Evaluates classification accuracy and attack success rate for **both English and Chinese CLIP models**
on CIFAR-10 (balanced 1000-image sample, 100/class).

Results saved to `results/attack/`.

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'open_clip_torch', 'transformers', 'datasets',
                'matplotlib', 'Pillow'], check=False)

CompletedProcess(args=['d:\\ian\\2026summer\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-q', 'open_clip_torch', 'transformers', 'datasets', 'matplotlib', 'Pillow'], returncode=0)

In [2]:
import importlib, sys, os, platform, random, json, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import cm
import torch
import torch.nn.functional as F
import open_clip
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from datasets import load_dataset
from transformers import ChineseCLIPModel, ChineseCLIPProcessor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

LANGS = ['en', 'zh']
CLASSES = {
    'en': ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck'],
    'zh': ['飞机', '汽车', '鸟', '猫', '鹿', '狗', '青蛙', '马', '船', '卡车'],
}
TMPL = {'en': 'a photo of a {}.', 'zh': '一张{}的照片。'}

RESULTS_DIR = 'results/attack'
os.makedirs(RESULTS_DIR, exist_ok=True)
SETUP_LABEL  = 'unilingual'
ATTACK_LABEL = 'unilingual'

Device: cuda


## 1. Model definitions

In [3]:
def classify(model, imgs, words, batch_size=128):
    preds = []
    for i in range(0, len(imgs), batch_size):
        imf = model.embed_images(imgs[i:i + batch_size])
        tf  = model.embed_texts(words)
        preds.append((imf @ tf.t()).argmax(-1).cpu().numpy())
    return np.concatenate(preds)

def classify_conf(model, imgs, words, batch_size=128):
    """Returns (predictions, max-cosine-sim confidences)."""
    preds, confs = [], []
    for i in range(0, len(imgs), batch_size):
        imf = model.embed_images(imgs[i:i + batch_size])
        tf  = model.embed_texts(words)
        sims = imf @ tf.t()
        preds.append(sims.argmax(-1).cpu().numpy())
        confs.append(sims.max(-1).values.cpu().numpy())
    return np.concatenate(preds), np.concatenate(confs)

def _clip_feat(out):
    if torch.is_tensor(out): return out
    if getattr(out, 'pooler_output', None) is not None: return out.pooler_output
    raise TypeError(type(out))

class EnCLIP:
    lang = 'en'
    def __init__(self):
        self.m, _, self.pp = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
        self.m = self.m.to(DEVICE).eval()
        self.tok = open_clip.get_tokenizer('ViT-B-32')
    @torch.no_grad()
    def embed_images(self, imgs):
        x = torch.stack([self.pp(im) for im in imgs]).to(DEVICE)
        return F.normalize(self.m.encode_image(x), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.tok([TMPL['en'].format(w) for w in words]).to(DEVICE)
        return F.normalize(self.m.encode_text(t), dim=-1)

class ZhCLIP:
    lang = 'zh'
    def __init__(self):
        self.m = ChineseCLIPModel.from_pretrained('OFA-Sys/chinese-clip-vit-base-patch16').to(DEVICE).eval()
        self.p = ChineseCLIPProcessor.from_pretrained('OFA-Sys/chinese-clip-vit-base-patch16')
    @torch.no_grad()
    def embed_images(self, imgs):
        pv = self.p(images=imgs, return_tensors='pt').pixel_values.to(DEVICE)
        return F.normalize(_clip_feat(self.m.get_image_features(pixel_values=pv)), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.p(text=[TMPL['zh'].format(w) for w in words], padding=True, return_tensors='pt').to(DEVICE)
        out = self.m.get_text_features(input_ids=t['input_ids'], attention_mask=t['attention_mask'],
                                        token_type_ids=t.get('token_type_ids'))
        return F.normalize(_clip_feat(out), dim=-1)

MODEL_CLS = {'en': EnCLIP, 'zh': ZhCLIP}
print('Model classes defined:', list(MODEL_CLS.keys()))

Model classes defined: ['en', 'zh']


## 2. Attack helpers

In [4]:
DISPLAY_SIZE = 224
NUM_BOXES    = 2
FONT_SIZE    = 24
PAD          = 8
_FONT_CACHE  = {}

def _font_paths():
    if platform.system() == 'Windows':
        winfonts = os.path.join(os.environ.get('WINDIR', r'C:\Windows'), 'Fonts')
        cjk   = os.path.join(winfonts, 'msyh.ttc')
        latin = os.path.join(winfonts, 'arial.ttf')
        if not os.path.exists(latin): latin = cjk
        return (cjk if os.path.exists(cjk) else None,
                latin if os.path.exists(latin) else None)
    for d in ['/usr/share/fonts', '/Library/Fonts', os.path.expanduser('~/.fonts')]:
        for f in ['NotoSansCJK-Regular.ttc', 'NotoSans-Regular.ttf']:
            p = os.path.join(d, f)
            if os.path.exists(p): return p, p
    return None, None

_CJK_FONT, _LAT_FONT = _font_paths()

def _font_for(word):
    return _CJK_FONT if any(ord(c) > 127 for c in word) else _LAT_FONT

def _get_font(fp, size=FONT_SIZE):
    key = (fp or '__default__', size)
    if key not in _FONT_CACHE:
        try:    _FONT_CACHE[key] = ImageFont.truetype(fp, size) if fp else ImageFont.load_default()
        except: _FONT_CACHE[key] = ImageFont.load_default()
    return _FONT_CACHE[key]

def _rects_overlap(a, b):
    return not (a[2] <= b[0] or b[2] <= a[0] or a[3] <= b[1] or b[3] <= a[1])

def _random_nonoverlapping_rect(rng, box_w, box_h, placed):
    x_hi = max(0, DISPLAY_SIZE - box_w)
    y_hi = max(0, DISPLAY_SIZE - box_h)
    rect_x, rect_y = 0, 0
    for _ in range(64):
        rect_x = rng.randint(0, x_hi) if x_hi > 0 else 0
        rect_y = rng.randint(0, y_hi) if y_hi > 0 else 0
        rect = (rect_x, rect_y, rect_x + box_w, rect_y + box_h)
        if all(not _rects_overlap(rect, p) for p in placed): return rect
    return (rect_x, rect_y, rect_x + box_w, rect_y + box_h)

def _draw_text_box(draw, word, rect, font):
    rx, ry, rx2, ry2 = rect
    bb = draw.textbbox((0, 0), word, font=font)
    draw.rectangle([rx, ry, rx2, ry2], fill='white')
    draw.text((rx + PAD - bb[0], ry + PAD - bb[1]), word, fill='black', font=font)

print('Font paths:', _CJK_FONT, _LAT_FONT)

Font paths: C:\WINDOWS\Fonts\msyh.ttc C:\WINDOWS\Fonts\arial.ttf


In [5]:
def draw_word(img, word, img_idx, already_224=False):
    """Place the same word in NUM_BOXES non-overlapping positions (all English)."""
    fp = _font_for(word)
    if not already_224:
        img = img.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC)
    else:
        img = img.copy()
    font  = _get_font(fp)
    draw  = ImageDraw.Draw(img)
    bb    = draw.textbbox((0, 0), word, font=font)
    bw    = (bb[2] - bb[0]) + 2 * PAD
    bh    = (bb[3] - bb[1]) + PAD + 12
    placed = []
    for box_i in range(NUM_BOXES):
        rng  = random.Random(int(img_idx) * NUM_BOXES + box_i)
        rect = _random_nonoverlapping_rect(rng, bw, bh, placed)
        placed.append(rect)
        _draw_text_box(draw, word, rect, font)
    return img

def build_attacked_images(base_imgs, img_indices, attack_lang, n_workers=None):
    """Unilingual attack: same word in both boxes."""
    words     = [CLASSES[attack_lang][target[int(k)]] for k in img_indices]
    n_workers = n_workers or min(8, os.cpu_count() or 4)
    def _one(args):
        im, word, img_idx = args
        return draw_word(im, word, img_idx=int(img_idx), already_224=True)
    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        return list(pool.map(_one, zip(base_imgs, words, img_indices)))

print(f'Unilingual attack helper ready: {NUM_BOXES} boxes @ size {FONT_SIZE} (both EN)')

Unilingual attack helper ready: 2 boxes @ size 24 (both EN)


## 3. Dataset

In [6]:
hf = load_dataset('uoft-cs/cifar10', split='test')
label_key = 'label' if 'label' in hf.column_names else 'labels'
image_key = 'img'   if 'img'   in hf.column_names else 'image'

_sample_path = '../../image_samples/CIFAR10_BALANCED_1000_SAMPLE.json'
with open(_sample_path, encoding='utf-8') as f:
    _saved = json.load(f)

idx  = _saved['idx']
rows = hf.select(idx)
true = np.array(rows[label_key])
assert len(idx) == 1000
assert np.array_equal(true, np.array(_saved['true']))
assert all((true == c).sum() == 100 for c in range(10))

rng    = random.Random(0)
target = np.array([rng.choice([c for c in range(10) if c != int(true[k])])
                   for k in range(len(idx))])

clean     = [im.convert('RGB') for im in rows[image_key]]
print('Upscaling clean images to 224px...', end=' ', flush=True)
t0 = time.time()
clean_224 = [im.resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC) for im in clean]
print(f'{time.time()-t0:.1f}s')

all_idx  = np.arange(len(clean))
tune_idx = np.concatenate([np.where(true == c)[0][:10] for c in range(10)])
print(f'Loaded {len(clean)} images; tune subset = {len(tune_idx)}')

Upscaling clean images to 224px... 0.2s
Loaded 1000 images; tune subset = 100


## 4. Load models

In [7]:
models = {}
for lang, cls in MODEL_CLS.items():
    t0 = time.time()
    print(f'Loading {lang}...', end=' ', flush=True)
    models[lang] = cls()
    print(f'{time.time()-t0:.1f}s')

# Standard text embeddings: each model with its own language
TEXT_EMB = {lang: models[lang].embed_texts(CLASSES[lang]).detach() for lang in LANGS}

clean_preds = {lang: classify(models[lang], clean_224, CLASSES[lang]) for lang in LANGS}
clean_acc   = {lang: float((clean_preds[lang] == true).mean()) for lang in LANGS}
print('Clean acc:', {k: f'{100*v:.1f}%' for k, v in clean_acc.items()})

Loading en... 

d:\ian\2026summer\.venv\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


1.1s
Loading zh... 

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

1.4s
Clean acc: {'en': '85.9%', 'zh': '91.4%'}


## 5. Build attacked images & classify

In [8]:
print('Building Unilingual attacked images...')
t0 = time.time()
attacked_imgs = build_attacked_images(clean_224, all_idx, 'en')
print(f'Done in {time.time()-t0:.1f}s')

preds_attacked = {}
for ml in LANGS:
    preds_attacked[ml] = classify(models[ml], attacked_imgs, CLASSES[ml])

acc_atk  = {ml: float((preds_attacked[ml] == true).mean())   for ml in LANGS}
asr_atk  = {ml: float((preds_attacked[ml] == target).mean()) for ml in LANGS}

print('\nAttack results:')
print(f'  Clean acc    EN={100*clean_acc["en"]:.1f}%  ZH={100*clean_acc["zh"]:.1f}%')
print(f'  Attacked acc EN={100*acc_atk["en"]:.1f}%  ZH={100*acc_atk["zh"]:.1f}%')
print(f'  ASR          EN={100*asr_atk["en"]:.1f}%  ZH={100*asr_atk["zh"]:.1f}%')

Building Unilingual attacked images...
Done in 0.2s

Attack results:
  Clean acc    EN=85.9%  ZH=91.4%
  Attacked acc EN=3.4%  ZH=27.1%
  ASR          EN=96.5%  ZH=71.8%


## 6. Per-class accuracy breakdown

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
class_names = CLASSES['en']
x = np.arange(len(class_names))
for ax, ml in zip(axes, LANGS):
    clean_pc  = np.array([(clean_preds[ml][true == c] == c).mean() for c in range(10)])
    attack_pc = np.array([(preds_attacked[ml][true == c] == c).mean() for c in range(10)])
    ax.bar(x - 0.2, clean_pc  * 100, 0.4, label='clean',   color='steelblue')
    ax.bar(x + 0.2, attack_pc * 100, 0.4, label='attacked', color='tomato')
    ax.set_xticks(x); ax.set_xticklabels(class_names, rotation=40, ha='right', fontsize=8)
    ax.set_title(f'{ml.upper()} model — per-class accuracy')
    ax.set_ylabel('%'); ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('Clean vs attacked per-class accuracy', fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved -> per_class_accuracy.png')

Saved -> per_class_accuracy.png


## 7. GradCAM heatmaps (sample)

In [10]:
def _norm_cam(cam):
    cam = cam.relu() if isinstance(cam, torch.Tensor) else np.maximum(cam, 0)
    cam = cam.detach().cpu().numpy() if isinstance(cam, torch.Tensor) else cam
    cam = cam - cam.min()
    mx  = cam.max()
    return cam / mx if mx > 0 else cam

def _cam_from_conv(act, grad):
    w = grad.mean(dim=(2, 3), keepdim=True)
    return _norm_cam((w * act).sum(dim=1).squeeze(0))

def gradcam_en(pil_img, target_idx):
    wrapper = models['en']; acts = {}
    def hook(_m, _i, out): out.retain_grad(); acts['v'] = out
    handle = wrapper.m.visual.conv1.register_forward_hook(hook)
    x        = wrapper.pp(pil_img).unsqueeze(0).to(DEVICE)
    feat     = wrapper.m.visual(x)
    img_feat = F.normalize(feat, dim=-1)
    score    = (img_feat @ TEXT_EMB['en'][target_idx:target_idx+1].T).squeeze()
    wrapper.m.zero_grad(); score.backward()
    cam = _cam_from_conv(acts['v'].detach(), acts['v'].grad)
    handle.remove(); return cam

def gradcam_zh(pil_img, target_idx):
    wrapper = models['zh']; acts = {}
    patch  = wrapper.m.vision_model.embeddings.patch_embedding
    def hook(_m, _i, out): out.retain_grad(); acts['v'] = out
    handle = patch.register_forward_hook(hook)
    pv     = wrapper.p(images=[pil_img], return_tensors='pt').pixel_values.to(DEVICE)
    out    = wrapper.m.get_image_features(pixel_values=pv)
    img_feat = F.normalize(_clip_feat(out), dim=-1)
    score  = (img_feat @ TEXT_EMB['zh'][target_idx:target_idx+1].T).squeeze()
    wrapper.m.zero_grad(); score.backward()
    cam = _cam_from_conv(acts['v'].detach(), acts['v'].grad)
    handle.remove(); return cam

def overlay_cam(pil_img, cam, alpha=0.50):
    cam_img = Image.fromarray((cam * 255).astype(np.uint8)).resize(
        (DISPLAY_SIZE, DISPLAY_SIZE), Image.BILINEAR)
    heat  = cm.jet(np.array(cam_img) / 255.0)[:, :, :3]
    base  = np.array(pil_img.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE))).astype(np.float32) / 255.0
    blend = np.clip((1 - alpha) * base + alpha * heat, 0, 1)
    return Image.fromarray((blend * 255).astype(np.uint8))

GRADCAM_FN = {'en': gradcam_en, 'zh': gradcam_zh}
print('Standard GradCAM helpers ready.')

Standard GradCAM helpers ready.


In [11]:
TEXT_EMB  # already defined in load-models cell

# Visualise GradCAM on 5 samples (one per row: attacked, EN cam, ZH cam)
sample_pos = [int(np.where(true == c)[0][0]) for c in range(5)]
fig, axes  = plt.subplots(5, 3, figsize=(9, 17))
col_titles = ['Attacked', 'EN GradCAM', 'ZH GradCAM']
for ax, t_title in zip(axes[0], col_titles):
    ax.set_title(t_title, fontsize=10, fontweight='bold')

for row_i, pos in enumerate(sample_pos):
    img   = attacked_imgs[pos]
    pred_en = int(preds_attacked['en'][pos])
    pred_zh = int(preds_attacked['zh'][pos])
    cam_en  = gradcam_en(img, pred_en)
    cam_zh  = gradcam_zh(img, pred_zh)
    for col_i, panel in enumerate([img, overlay_cam(img, cam_en), overlay_cam(img, cam_zh)]):
        ax = axes[row_i, col_i]
        ax.imshow(panel); ax.axis('off')
    axes[row_i, 0].set_ylabel(CLASSES['en'][true[pos]], fontsize=8, rotation=0, labelpad=36, va='center')

plt.suptitle('GradCAM on attacked images', fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/gradcam_heatmaps.png', dpi=140, bbox_inches='tight')
plt.close()
print('Saved -> gradcam_heatmaps.png')

d:\ian\2026summer\.venv\Lib\site-packages\torch\autograd\graph.py:882: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\cuda\CublasHandlePool.cpp:370.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Saved -> gradcam_heatmaps.png


## 8. Sample visualisation

In [12]:
sample_idx = [int(np.where(true == c)[0][0]) for c in range(5)]
fig, axes  = plt.subplots(5, 2, figsize=(7, 17))
for row_i, pos in enumerate(sample_idx):
    for col_i, img in enumerate([clean_224[pos], attacked_imgs[pos]]):
        axes[row_i, col_i].imshow(img); axes[row_i, col_i].axis('off')
        axes[row_i, col_i].set_title(['Clean', 'Attacked'][col_i], fontsize=9)
    axes[row_i, 0].set_ylabel(CLASSES['en'][true[pos]], fontsize=8, rotation=0, labelpad=36, va='center')
plt.suptitle('Unilingual sample visualisation', fontsize=11)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/sample_viz.png', dpi=140, bbox_inches='tight')
plt.close()
print('Saved -> sample_viz.png')

Saved -> sample_viz.png


## 9. Save results JSON

In [13]:
results = {
    'setup':          SETUP_LABEL,
    'method':         'no_defense',
    'attack':         ATTACK_LABEL,
    'sample':         'CIFAR10_BALANCED_1000_SAMPLE',
    'n_images':       len(all_idx),
    'inference_cost': 2,
    'clean_acc':      clean_acc,
    'attacked_acc':   acc_atk,
    'attacked_asr':   asr_atk,
    'attacked_acc_mean': float(np.mean(list(acc_atk.values()))),
}
out_path = f'{RESULTS_DIR}/confusion_results.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f'Saved -> {out_path}')

Saved -> results/attack/confusion_results.json
